
# DDRI 군집별 1차 결과 취합 노트북

- 목적: `cluster00 ~ cluster04` 1차 실험 결과를 한 자리에서 취합해 `ddri_cluster_model_metrics_collection_template.csv`를 재생성한다.
- 입력: `works/05_prediction_long/output/team_cluster_template_outputs/*.csv`
- 출력: `summary_aggregation/output/data/ddri_cluster_model_metrics_collection_template.csv`


In [1]:

from pathlib import Path
import pandas as pd

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
AGG_DIR = ROOT / 'works/05_prediction_long/cheng80/summary_aggregation'
OUTPUT_DATA_DIR = AGG_DIR / 'output/data'
INPUT_DIR = ROOT / 'works/05_prediction_long/output/team_cluster_template_outputs'
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

cluster_meta = {
    '업무/상업 혼합형': ('cluster00', 'works/05_prediction_long/cheng80/cluster00/01_cluster_modeling.ipynb'),
    '아침 도착 업무 집중형': ('cluster01', 'works/05_prediction_long/cheng80/cluster01/01_cluster_modeling.ipynb'),
    '주거 도착형': ('cluster02', 'works/05_prediction_long/cheng80/cluster02/01_cluster_modeling.ipynb'),
    '생활권 혼합형': ('cluster03', 'works/05_prediction_long/cheng80/cluster03/01_cluster_modeling.ipynb'),
    '외곽 주거형': ('cluster04', 'works/05_prediction_long/cheng80/cluster04/01_cluster_modeling.ipynb'),
}


## 1. 팀 템플릿 1차 결과를 읽어 취합표 생성

In [2]:

rows = []
for csv_path in sorted(INPUT_DIR.glob('ddri_*_model_metrics.csv')):
    df = pd.read_csv(csv_path, encoding='utf-8-sig')
    station_group = df['station_group'].iloc[0]
    cluster_code, notebook_path = cluster_meta[station_group]
    best_model = (
        df[df['split'] == 'validation_2024']
        .sort_values('rmse', ascending=True)
        .iloc[0]['model']
    )
    for _, r in df.iterrows():
        rows.append({
            'owner': 'cheng80',
            'cluster_code': cluster_code,
            'station_group': r['station_group'],
            'model': r['model'],
            'split': r['split'],
            'rmse': round(float(r['rmse']), 4),
            'mae': round(float(r['mae']), 4),
            'r2': round(float(r['r2']), 4),
            'best_model_flag': int(r['split'] == 'test_2025_refit' and r['model'] == best_model),
            'notebook_path': notebook_path,
            'notes': '',
        })

out_df = pd.DataFrame(rows).sort_values(['cluster_code', 'split', 'model']).reset_index(drop=True)
out_path = OUTPUT_DATA_DIR / 'ddri_cluster_model_metrics_collection_template.csv'
out_df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(out_df.to_string(index=False))
print() 
print('saved:', out_path)


  owner cluster_code station_group            model           split   rmse    mae     r2  best_model_flag                                                        notebook_path notes
cheng80    cluster00     업무/상업 혼합형    LightGBM_RMSE test_2025_refit 0.8113 0.5439 0.3212                1 works/05_prediction_long/cheng80/cluster00/01_cluster_modeling.ipynb      
cheng80    cluster00     업무/상업 혼합형    LightGBM_RMSE validation_2024 0.8987 0.6092 0.3054                0 works/05_prediction_long/cheng80/cluster00/01_cluster_modeling.ipynb      
cheng80    cluster00     업무/상업 혼합형 LinearRegression validation_2024 0.9207 0.6377 0.2708                0 works/05_prediction_long/cheng80/cluster00/01_cluster_modeling.ipynb      
cheng80    cluster01  아침 도착 업무 집중형    LightGBM_RMSE test_2025_refit 1.3462 0.8042 0.6398                1 works/05_prediction_long/cheng80/cluster01/01_cluster_modeling.ipynb      
cheng80    cluster01  아침 도착 업무 집중형    LightGBM_RMSE validation_2024 1.5660 0.9267 0.6542       